In [0]:
import joblib
import pandas as pd
import numpy as np

### Load model

In [0]:
model = joblib.load('/Workspace/Users/ajiboyeniola@gmail.com/lead-scoring/models/lead_scoring_model.pkl')

print(f"Model loaded successfully")
print(f"Model type: {type(model).__name__}")

In [0]:
print(type(model))                          # Pipeline
print(model.named_steps)                    # shows preprocessing + model steps
print(type(model.named_steps['model']))     # LogisticRegression

### Score the full dataset

In [0]:
X_test = pd.read_csv("/Workspace/Users/ajiboyeniola@gmail.com/lead-scoring/data/processed/X_test.csv")
y_test = pd.read_csv("/Workspace/Users/ajiboyeniola@gmail.com/lead-scoring/data/processed/y_test.csv")

# Score the test set
y_prob_test = model.predict_proba(X_test)[:, 1]



In [0]:
scored_leads = pd.DataFrame({
    'conversion_probability': y_prob_test.round(4),
    'actual_converted':       y_test['converted'].values
})

# Assign tiers
def assign_tier(prob):
    if prob >= 0.70:
        return 'High Priority'
    elif prob >= 0.50:
        return 'Medium Priority'
    else:
        return 'Low Priority'

# Assign recommended actions
def assign_action(tier):
    actions = {
        'High Priority':   'Call immediately',
        'Medium Priority': 'Add to email nurture campaign',
        'Low Priority':    'Deprioritize'
    }
    return actions[tier]

scored_leads['tier']                = scored_leads['conversion_probability'].apply(assign_tier)
scored_leads['recommended_action']  = scored_leads['tier'].apply(assign_action)
scored_leads['expected_value']      = (scored_leads['conversion_probability'] * 560).round(2)

In [0]:
# Reset index to align correctly
X_test_reset = X_test.reset_index(drop=True)

# Combine features with scores
final_output = pd.concat([X_test_reset, scored_leads], axis=1)

# Sort by expected value — highest value leads first
final_output = final_output.sort_values('expected_value', ascending=False).reset_index(drop=True)

In [0]:
# Make sure everything looks right before saving
print(f"Total leads scored: {len(final_output)}")
print(f"\nTier distribution:")
print(final_output['tier'].value_counts())
print(f"\nTop 5 highest priority leads:")
print(final_output[['conversion_probability', 'tier', 'expected_value', 'recommended_action']].head())
print(f"\nAny nulls in output?")
print(final_output.isnull().sum()[final_output.isnull().sum() > 0])

In [0]:
final_output.to_csv('/Workspace/Users/ajiboyeniola@gmail.com/lead-scoring/data/outputs/scored_leads.csv', index=False)
print(f"Saved {len(final_output)} scored leads to outputs/scored_leads.csv")